# Tutorial 11 — Full-Text RAG Pipeline over Scientific Literature
**Author:** Himanshu Goel | [Website](https://hgoelgithub.github.io)

Retrieval-Augmented Generation (RAG) combines a vector database of documents with an LLM to answer questions grounded in retrieved context.

This notebook goes beyond vanilla abstract-only RAG:
- **Full-text fetching** — pulls complete articles from PubMed Central (PMC) for open-access papers, falling back to abstract
- **Chunking** — splits long articles so each retrieved chunk stays focused
- **Hybrid search** — BM25 keyword matching + semantic embeddings, ensembled for better recall

In [ ]:
# ── Core LangChain packages ───────────────────────────────────────────────────
# langchain          : orchestration framework (chains, runnables, prompts)
# langchain-community: third-party integrations (BM25Retriever, Chroma wrapper)
# langchain-openai   : ChatOpenAI wrapper for GPT models
!pip install langchain langchain-community langchain-openai -q

# ── Vector store & embedding model ───────────────────────────────────────────
# chromadb           : fast in-process vector database for storing embeddings
# sentence-transformers: pretrained models that convert text → fixed-size vectors
!pip install chromadb sentence-transformers -q

# ── Utilities ────────────────────────────────────────────────────────────────
# langchain-huggingface   : HuggingFaceEmbeddings wrapper (uses sentence-transformers)
# langchain-text-splitters: splits long documents into overlapping chunks
# rank_bm25               : BM25 keyword-based ranking algorithm (no GPU needed)
!pip install langchain-huggingface langchain-text-splitters rank_bm25 -q

# ── HTTP + transformers ───────────────────────────────────────────────────────
# requests     : HTTP calls to the NCBI PubMed / PMC E-utilities API
# transformers : required by sentence-transformers; pinned below 5.x to avoid
#                a known nn import bug introduced in transformers 5.6
!pip install requests -q
!pip install "transformers>=4.41.0,<5.0" -q

import transformers; print("transformers", transformers.__version__)


## Step 1 — Fetch PubMed full text (PMC) with abstract fallback

In [ ]:
import requests                        # HTTP requests to NCBI API
import xml.etree.ElementTree as ET     # parse PubMed/PMC XML responses
import time                            # rate-limit compliance (sleep between requests)
from langchain_core.documents import Document  # LangChain's standard document container

# ─────────────────────────────────────────────────────────────────────────────
# fetch_pubmed_fulltext
#
# Queries PubMed for articles matching `query`, then tries to retrieve the
# complete article body from PubMed Central (PMC) for open-access papers.
# If full text is unavailable (paywalled), it falls back to the abstract.
#
# Returns a list of LangChain Document objects, each holding either:
#   • full article text (title + section-by-section body), or
#   • abstract text
# ─────────────────────────────────────────────────────────────────────────────
def fetch_pubmed_fulltext(query: str, max_results: int = 20) -> list[Document]:
    # NCBI E-utilities base URL — all PubMed/PMC API calls use this prefix
    base = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

    # ── Step 1: Search PubMed for article IDs (PMIDs) ────────────────────────
    # esearch.fcgi returns a JSON list of PMIDs matching the search term
    search = requests.get(
        f"{base}esearch.fcgi",
        params={"db": "pubmed", "term": query, "retmax": max_results, "retmode": "json"}
    )
    pmids = search.json()["esearchresult"]["idlist"]
    if not pmids:
        return []   # no results found for this query

    # ── Step 2: Fetch metadata (title, abstract, PMC ID) for all PMIDs ───────
    # efetch.fcgi with rettype=abstract returns structured XML with full metadata
    fetch = requests.get(
        f"{base}efetch.fcgi",
        params={"db": "pubmed", "id": ",".join(pmids), "rettype": "abstract", "retmode": "xml"}
    )
    root = ET.fromstring(fetch.text)

    # Parse each article's metadata into a lookup dict keyed by PMID
    abstract_map = {}
    for art in root.iter("PubmedArticle"):
        pmid  = art.findtext(".//PMID", "")
        # itertext() flattens nested XML tags (e.g. italics inside titles)
        title = "".join("".join(t.itertext()) for t in art.iter("ArticleTitle"))
        abstr = "".join("".join(t.itertext()) for t in art.iter("AbstractText"))
        year  = art.findtext(".//PubDate/Year", "?")
        # PMC ID (e.g. "PMC1234567") is only present for open-access articles
        pmcid = art.findtext(".//ArticleId[@IdType='pmc']", "")
        abstract_map[pmid] = {"title": title, "abstract": abstr, "year": year, "pmcid": pmcid}

    docs     = []
    ft_count = 0   # counter: how many articles have full text vs. abstract-only

    for pmid, meta in abstract_map.items():
        content = None
        source  = "abstract"   # assume abstract until full text is confirmed

        # ── Step 3: Try to fetch full article text from PMC ──────────────────
        # PMC hosts the complete XML body of open-access papers.
        # We iterate over <sec> (section) elements and concatenate their text.
        if meta["pmcid"]:
            try:
                pmc = requests.get(
                    f"{base}efetch.fcgi",
                    params={"db": "pmc", "id": meta["pmcid"], "rettype": "xml", "retmode": "xml"},
                    timeout=20
                )
                pmc_root = ET.fromstring(pmc.text)
                sections = []
                for sec in pmc_root.iter("sec"):
                    sec_title = sec.findtext("title", "").strip()
                    # Concatenate all <p> (paragraph) elements within this section
                    sec_text  = " ".join("".join(p.itertext()) for p in sec.iter("p")).strip()
                    if sec_text:
                        header = f"[{sec_title}]" if sec_title else "[Section]"
                        sections.append(f"{header}\n{sec_text}")

                if sections:
                    # Build a single string: article title + all section bodies
                    content  = f"Title: {meta['title']}\n\n" + "\n\n".join(sections)
                    source   = "fulltext"
                    ft_count += 1

                # NCBI allows ≤3 unauthenticated requests/second; sleep to stay compliant
                time.sleep(0.35)
            except Exception:
                pass   # network error or malformed XML → fall through to abstract

        # ── Step 4: Fall back to abstract if full text is unavailable ─────────
        if content is None:
            if meta["abstract"].strip():
                content = f"Title: {meta['title']}\n\nAbstract: {meta['abstract']}"
            else:
                continue   # skip articles with no text at all

        # Wrap content + metadata in a LangChain Document for downstream use
        docs.append(Document(
            page_content=content,
            metadata={
                "pmid" : pmid,
                "pmcid": meta["pmcid"],
                "title": meta["title"],
                "year" : meta["year"],
                "source": source,   # "fulltext" or "abstract" — useful for filtering
            }
        ))

    print(f"Fetched {len(docs)} docs — {ft_count} full-text, {len(docs)-ft_count} abstract-only")
    return docs


# ── Run the fetch ─────────────────────────────────────────────────────────────
# Searching for SILCS-related papers; max_results caps the number of PMIDs queried
docs = fetch_pubmed_fulltext("SILCS protein ligand binding drug discovery", max_results=15)

# Preview the first 5 documents to confirm the fetch worked
for d in docs[:5]:
    print(f"  [{d.metadata['year']}] [{d.metadata['source']:8s}] PMID {d.metadata['pmid']}: {d.metadata['title'][:60]}...")


## Step 2 — Chunk documents and embed into ChromaDB

In [ ]:
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# ── Step 1: Split documents into chunks ──────────────────────────────────────
# Full-text articles can be 10,000+ words. Embedding models have a token limit
# (~512 tokens), and long chunks produce blurry embeddings that hurt retrieval.
# RecursiveCharacterTextSplitter splits on paragraph/sentence/word boundaries
# in order, keeping chunks as semantically coherent as possible.
#
#   chunk_size=800   → each chunk is at most 800 characters (~150 words)
#   chunk_overlap=100→ 100-char overlap between adjacent chunks so context
#                      at chunk boundaries is not lost
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=100)
chunks   = splitter.split_documents(docs)   # preserves metadata from each Document

print(f"{len(docs)} documents → {len(chunks)} chunks")
for c in chunks[:2]:
    print(f"  [{c.metadata.get('source','?')}] {c.page_content[:120]}...\n")

# ── Step 2: Load an embedding model ──────────────────────────────────────────
# Embeddings convert text into dense numeric vectors so we can measure
# semantic similarity (e.g. "binding affinity" ≈ "binding free energy").
# all-MiniLM-L6-v2 is small (22M params), fast, and good enough for retrieval.
# It runs locally — no API key or internet connection needed after download.
embedder = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# ── Step 3: Build the vector store ───────────────────────────────────────────
# ChromaDB stores each chunk as a vector (embedding) alongside its raw text.
# EphemeralClient = purely in-memory, which avoids SQLite file-lock problems
# if you re-run the cell or run multiple kernels simultaneously.
# Chroma.from_documents embeds every chunk and inserts them in one call.
chroma_client = chromadb.EphemeralClient()
vectordb      = Chroma.from_documents(chunks, embedder, client=chroma_client)
print(f"Indexed {len(chunks)} chunks into ChromaDB (in-memory)")

# ── Sanity check: run a test semantic query ───────────────────────────────────
# similarity_search converts the query to a vector and returns the k nearest
# chunks by cosine distance — no keyword matching, purely meaning-based.
query     = "protein-ligand binding affinity prediction"
retrieved = vectordb.similarity_search(query, k=3)
print(f"\nTop 3 semantic hits for: '{query}'")
for r in retrieved:
    print(f"  [{r.metadata.get('year','?')}] [{r.metadata.get('source','?')}] {r.metadata.get('title','')[:60]}...")


## Step 3 — Hybrid RAG Q&A (BM25 + semantic ensemble)

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()   # reads OPENAI_API_KEY from a .env file in the project root

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.retrievers import BM25Retriever
from collections import defaultdict

# ── LLM setup ────────────────────────────────────────────────────────────────
# gpt-4o-mini is fast and cost-effective for RAG answers.
# temperature=0 makes outputs deterministic (no randomness in word choice).
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ─────────────────────────────────────────────────────────────────────────────
# SimpleEnsembleRetriever — Hybrid BM25 + Semantic retrieval
#
# WHY hybrid?
#   • BM25 (keyword-based): excels at exact term matching, e.g. "SILCS score".
#     Fast, no embeddings needed. Misses synonyms and paraphrases.
#   • Semantic (embedding-based): finds conceptually related chunks even when
#     the exact words differ, e.g. "binding free energy" ≈ "affinity score".
#     Slower, but catches paraphrases BM25 would miss.
#
# Combining them (40% BM25 + 60% semantic) improves recall over either alone.
#
# Scoring: each retriever returns a ranked list. We use reciprocal rank fusion:
#   score += weight / (rank + 1)
# so a document ranked #1 by both retrievers scores higher than one ranked
# #1 by only one — the ensemble naturally promotes documents that both agree on.
# ─────────────────────────────────────────────────────────────────────────────
class SimpleEnsembleRetriever:
    def __init__(self, bm25_retriever, semantic_retriever, weights=(0.4, 0.6)):
        self.bm25     = bm25_retriever
        self.semantic = semantic_retriever
        self.w_bm25, self.w_semantic = weights

    def invoke(self, query: str) -> list:
        # Retrieve top-k candidates from each retriever independently
        bm25_docs     = self.bm25.invoke(query)
        semantic_docs = self.semantic.invoke(query)

        # Accumulate reciprocal-rank scores for each unique chunk
        scores = defaultdict(float)
        for rank, doc in enumerate(bm25_docs):
            scores[doc.page_content] += self.w_bm25 / (rank + 1)
        for rank, doc in enumerate(semantic_docs):
            scores[doc.page_content] += self.w_semantic / (rank + 1)

        # Deduplicate (same chunk can appear in both lists) and sort by score
        all_docs    = {doc.page_content: doc for doc in bm25_docs + semantic_docs}
        sorted_keys = sorted(scores, key=lambda k: scores[k], reverse=True)
        return [all_docs[k] for k in sorted_keys[:8]]   # return top 8 combined


# ── Build the two individual retrievers ──────────────────────────────────────
# BM25Retriever indexes all chunk texts using term-frequency statistics.
# k=4 means it returns the 4 best keyword matches per query.
bm25_retriever   = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 4

# as_retriever wraps the ChromaDB vector store as a LangChain retriever.
# k=4 means it returns the 4 nearest embedding neighbours per query.
semantic_retriever = vectordb.as_retriever(search_kwargs={"k": 4})

# Combine into the hybrid retriever (returns up to 8 de-duplicated chunks)
hybrid_retriever = SimpleEnsembleRetriever(bm25_retriever, semantic_retriever, weights=(0.4, 0.6))


# ── Prompt template ───────────────────────────────────────────────────────────
# The prompt instructs the LLM to answer ONLY from the provided context.
# This keeps answers grounded in the retrieved papers and prevents hallucination.
# If the answer isn't in the context, the model is told to say so explicitly.
prompt = ChatPromptTemplate.from_template(
    "Use only the context below to answer the question. "
    "If the answer is not in the context, say 'Not found in retrieved papers.'\n\n"
    "Context:\n{context}\n\nQuestion: {question}"
)


# ── format_docs ───────────────────────────────────────────────────────────────
# Converts a list of Document objects into a single formatted string that is
# injected into the {context} slot of the prompt above.
# Each chunk is labeled with its PMID, year, and source so the LLM can
# reference specific papers in its answer.
def format_docs(docs: list) -> str:
    return "\n\n---\n\n".join(
        f"[PMID {d.metadata['pmid']} | {d.metadata['year']} | {d.metadata['source']}]\n{d.page_content}"
        for d in docs
    )


# ─────────────────────────────────────────────────────────────────────────────
# HybridQAChain — ties retrieval + formatting + LLM into one callable object
#
# invoke(query) flow:
#   1. hybrid_retriever.invoke(query) → list of top-8 Document chunks
#   2. format_docs(docs)              → single context string
#   3. prompt.invoke({context, question}) → formatted ChatPromptValue
#   4. llm.invoke(prompt)             → ChatMessage with the LLM's answer
#   5. StrOutputParser.invoke(msg)    → plain Python string
# ─────────────────────────────────────────────────────────────────────────────
class HybridQAChain:
    def __init__(self, prompt, llm, parser, retriever, formatter):
        # Build a LangChain pipe: prompt → LLM → string parser
        self.pipeline  = prompt | llm | parser
        self.retriever = retriever
        self.formatter = formatter

    def invoke(self, query: str) -> str:
        # Retrieve and format context, then run through the LLM pipeline
        docs    = self.retriever.invoke(query)
        context = self.formatter(docs)
        return self.pipeline.invoke({"context": context, "question": query})


# Assemble the full chain
chain = HybridQAChain(prompt, llm, StrOutputParser(), hybrid_retriever, format_docs)


# ── Run example questions ─────────────────────────────────────────────────────
questions = [
    "What is the SILCS method and how does it compute binding affinity?",
    "How does SILCS compare to FEP methods for protein-ligand binding?",
    "What is the application of SILCS to hERG cardiotoxicity prediction?",
]

for q in questions:
    print(f"Q: {q}")
    answer  = chain.invoke(q)
    sources = hybrid_retriever.invoke(q)   # re-retrieve to show provenance
    print(f"A: {answer[:400]}...")
    print(f"Sources ({len(sources)}): {[(d.metadata['pmid'], d.metadata['source']) for d in sources]}")
    print()


## Key takeaways
- **Full-text RAG** retrieves far richer context than abstract-only — methods, results, and discussion are now searchable
- **Chunking** (`chunk_size=800, overlap=100`) keeps each retrieval unit small enough for the embedding model to stay accurate
- **Hybrid search** (BM25 40% + semantic 60%) improves recall: BM25 catches exact keyword matches, semantic search catches paraphrases and synonyms
- **PMC fallback** — only open-access articles have full text; the pipeline gracefully degrades to abstract for paywalled papers
- For production: add a cross-encoder re-ranker, metadata filters (year, journal), and a larger embedding model (e.g. `BAAI/bge-large-en-v1.5`)